# Dog Breed Classifier — Colab Runner

Runs the Udacity AIPND *Pretrained Image Classifier* project end to end and
produces the three summary files required for submission:
`vgg_pet-images.txt`, `alexnet_pet-images.txt`, `resnet_pet-images.txt`.

**Run the cells in order, top to bottom.**

Two files are **not** in the repository and must come from the Udacity
project workspace (Step 3):

1. the 40 pet images (30 dog, 10 not-a-dog)
2. `imagenet1000_clsidx_to_labels.txt`


## Step 1 — Check the environment

Colab ships with PyTorch already installed, so nothing needs installing.
A GPU is optional; 40 images run fine on CPU in a couple of minutes.


In [ ]:
import torch, torchvision, sys
print('python     :', sys.version.split()[0])
print('torch      :', torch.__version__)
print('torchvision:', torchvision.__version__)
print('cuda       :', torch.cuda.is_available())


## Step 2 — Get the project code

Clones the repository. Make sure the fixed code has been pushed to GitHub
first, otherwise you will clone the old version.


In [ ]:
!git clone https://github.com/Hamdayrabby/1st_project_dog_breed_classifier.git
%cd 1st_project_dog_breed_classifier
!ls -la


## Step 3 — Upload the dataset and the ImageNet label map

From the Udacity workspace, download:

- the `pet_images` folder, zipped as **`pet_images.zip`**
- **`imagenet1000_clsidx_to_labels.txt`**

Then run this cell and select **both** files in the upload dialog.

> Use the original course images. Substitute or placeholder images make the
> results ungradeable — that is what caused the previous rejection.


In [ ]:
from google.colab import files
uploaded = files.upload()
print('
uploaded:', list(uploaded))


### Unpack the images

Handles a zip that contains a top-level `pet_images/` folder as well as one
that contains the images directly.


In [ ]:
import glob, os, shutil, zipfile

os.makedirs('pet_images', exist_ok=True)

with zipfile.ZipFile('pet_images.zip') as zf:
    zf.extractall('_unzipped')

# Move every image found anywhere in the archive into pet_images/
moved = 0
for path in glob.glob('_unzipped/**/*', recursive=True):
    if path.lower().endswith(('.jpg', '.jpeg', '.png')) and '__MACOSX' not in path:
        shutil.move(path, os.path.join('pet_images', os.path.basename(path)))
        moved += 1

shutil.rmtree('_unzipped', ignore_errors=True)
print(f'moved {moved} images into pet_images/')
print('pet_images now holds:', len(glob.glob('pet_images/*.jpg')), 'jpg files')


## Step 4 — Validate the dataset

This must print **`Dataset OK: 40 images (30 dog / 10 not-dog)`** before you
go any further. If it fails, fix the dataset — do not run the batch.


In [ ]:
!python validate_dataset.py --dir pet_images/ --dogfile dognames.txt


## Step 5 — Single-model smoke test

Confirms one architecture works before committing to all three. The first
run downloads the pretrained VGG16 weights (~528 MB), so expect a pause.


In [ ]:
!python check_images.py --dir pet_images/ --arch vgg --dogfile dognames.txt


## Step 6 — Run the full batch

Runs ResNet, AlexNet and VGG, writing one summary file each.


In [ ]:
!sh run_models_batch.sh


## Step 7 — Review the results

Sanity checks before you submit:

- `N Images` is **40**, `N Dog Images` **30**, `N Not-Dog Images` **10**
- runtime is **not** `00:00:00`
- the percentages are **not** all 100%

Expected figures for the course set:

| Metric | VGG | AlexNet | ResNet |
| :--- | :---: | :---: | :---: |
| % Match | ~87.5% | ~75% | ~82.5% |
| % Correct Dogs | 100% | 100% | 100% |
| % Correct Breed | ~93.3% | ~80% | ~90% |
| % Correct Not-Dogs | 100% | 100% | ~90% |


In [ ]:
for name in ['vgg', 'alexnet', 'resnet']:
    path = f'{name}_pet-images.txt'
    print('=' * 60)
    print(path)
    print('=' * 60)
    print(open(path).read())


## Step 8 — Download the result files

Save these three files, put them in your project folder, and resubmit.


In [ ]:
from google.colab import files

for name in ['vgg', 'alexnet', 'resnet']:
    files.download(f'{name}_pet-images.txt')
